# RecursiveCharacterTextSplitter

`RecursiveCharacterTextSplitter`는 **가장 권장되는 범용 텍스트 분할 방식**입니다.

## 주요 특징

- **재귀적 분할**: 여러 구분자를 우선순위에 따라 순차적으로 시도합니다.
- **계층적 접근**: 단락 → 문장 → 단어 순서로 분할하여 의미 단위를 보존합니다.
- **유연한 청크 크기**: 청크가 너무 크면 다음 구분자로 재귀적으로 재분할합니다.

## 동작 원리

1. **구분자 우선순위**: 기본값은 `["\n\n", "\n", " ", ""]`
   - `"\n\n"` (단락) → `"\n"` (줄) → `" "` (단어) → `""` (문자) 순서로 시도
2. **재귀적 분할**: 청크 크기가 `chunk_size`를 초과하면 다음 구분자로 재시도
3. **문맥 보존**: 가능한 한 큰 의미 단위를 유지하면서 분할

## CharacterTextSplitter와의 차이점

| 특징 | CharacterTextSplitter | RecursiveCharacterTextSplitter |
|------|----------------------|-------------------------------|
| 구분자 | 단일 구분자만 사용 | 여러 구분자를 순차적으로 시도 |
| 분할 방식 | 단순 분할 | 재귀적 분할 |
| 청크 크기 관리 | 초과 시 그대로 유지 | 초과 시 자동 재분할 |
| 권장 용도 | 단순 문서 | **대부분의 경우 (권장)** |

## 사용 시나리오

- ✅ **일반 텍스트 문서**: 대부분의 경우 최선의 선택
- ✅ **복잡한 문서 구조**: 단락, 문장, 단어 혼합 문서
- ✅ **의미 보존 중요**: 문맥을 유지하면서 분할이 필요할 때

# 02. RecursiveCharacterTextSplitter

## 학습 목표
- `RecursiveCharacterTextSplitter`의 재귀적 분할 원리를 이해해요
- 기본 구분자 우선순위(`\n\n` → `\n` → ` ` → `""`)의 동작 방식을 파악해요
- `CharacterTextSplitter`와의 차이를 실제 데이터로 비교해요

## 사전 지식
- 01-CharacterTextSplitter 완료

---

> **이전 노트북에서 배운 것**: CharacterTextSplitter는 단일 구분자만 사용해요. 청크가 너무 커도 자동으로 재분할하지 않아요.
>
> **이번 노트북에서 배울 것**: 여러 구분자를 **우선순위 순서**로 시도해서 의미 단위를 최대한 보존하며 분할해요. **대부분의 프로젝트에서 이 방식을 기본값으로 사용해요.**

## 샘플 데이터 로드

동일한 기술 용어 사전 데이터를 사용하여 RecursiveCharacterTextSplitter의 동작을 확인합니다.

In [1]:
# 샘플 데이터 파일을 읽어옵니다.
with open("./data/wings.txt", encoding="utf-8") as f:
    file = f.read()


로드된 텍스트의 일부를 확인합니다.

In [2]:
# 파일 내용 중 처음 500자를 출력합니다.

# 아래에 코드를 작성하세요
file[:500]
print(f' ==> [Line 4]: \033[38;2;117;173;103m[file[:500]]\033[0m({type(file[:500]).__name__}) = \033[38;2;214;21;174m{file[:500]}\033[0m')


 ==> [Line 4]: [file[:500]](str) = 박제(剝製)가 되어 버린 천재'를 아시오? 나는 유쾌하오. 이런 때 연애까지가 유쾌하오.

육신이 흐느적흐느적하도록 피로했을 때만 정신이 은화(銀貨)처럼 맑소. 니코틴이 내 횟배 앓는 뱃속으로 스미면 머리 속에 으례 백지가 준비되는 법이오. 그 위에다 나는 위트와 패러독스를 바둑 포석처럼 늘어놓았소. 가증할 상식의 병이오. 나는 또 여인과 생활을 설계하오. 연애 기법에마저 서먹서먹해진, 지성의 극치를 홀깃 좀 들여다본 일이 있는, 말하자면 일종의 정신분일자(精神奔逸者) 말이오. 이런 여인의 반(半) ― 그것은 온갖 것의 반이오 ― 만을 영수(領收)하는 생활을 설계한다는 말이오. 그런 생활 속에 한 발만 들여 놓고 흡사 두개의 태양처럼 마주 쳐다보면서 낄낄거리는 것이오. 나는 아마 어지간히 인생의 제행(諸行)이 싱거워서 견딜 수가 없게끔 되고 그만 둔 모양이요. 굿바이.

굿바이. 그대는 이따금 그대가 제일 싫어하는 음식을 탐식(貪食)하는 아이러니를 실천해 보는 것도 좋을 것 같소. 


## 1. 기본 사용법

`RecursiveCharacterTextSplitter`의 기본 설정으로 텍스트를 분할합니다.

### 주요 파라미터

- `chunk_size`: 청크의 최대 문자 수
- `chunk_overlap`: 청크 간 중복 영역
- `length_function`: 길이 계산 함수 (기본값: `len`)
- `is_separator_regex`: 구분자를 정규식으로 해석할지 여부 (기본값: `False`)
- `separators`: 분할에 사용할 구분자 리스트 (기본값: `["\n\n", "\n", " ", ""]`)

In [3]:
# ---------------------------------------------------
# RecursiveCharacterTextSplitter 생성
# ---------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1단계: RecursiveCharacterTextSplitter 생성
# - 기본 구분자 우선순위: ["\n\n", "\n", " ", ""]
#   단락 → 줄 → 단어 → 문자 순서로 시도하여 최적 분할점 탐색
# - chunk_size=250: 청크 최대 문자 수
# - chunk_overlap=50: 인접 청크 간 중복 (문맥 연결)

# 아래에 코드를 작성하세요
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False
)



## 2. 재귀적 분할 확인

텍스트를 분할하고 결과를 확인합니다. 첫 번째와 두 번째 청크를 비교하여 재귀적 분할이 어떻게 작동하는지 살펴봅니다.

In [5]:
# ============================================================
# TODO: RecursiveCharacterTextSplitter로 텍스트를 분할해보세요
# 힌트: text_splitter.create_documents([file]) → Document 리스트 반환
# 예상 결과: 30개의 Document가 생성됩니다
# ============================================================

# 1단계: 텍스트를 Document 객체 리스트로 분할
documents = text_splitter.create_documents([file])

print(len(documents))

print(len(documents[102].page_content))
print(documents[102].page_content)

142
86
제일 여기 시계가 어느 시계보다도 정확하리라는 것이 좋았다. 섣불리 서투른 시계를 보고 그것을 믿고 시간 전에 집에 돌아갔다가 큰 코를 다쳐서는 안 된다.


두 번째 청크를 확인하여 청크 간 중복(`chunk_overlap`)이 어떻게 작동하는지 살펴봅니다.

In [ ]:
# 두 번째 청크 확인

# 아래에 코드를 작성하세요


## 3. split_text() 메서드 사용

`split_text()` 메서드를 사용하면 순수 문자열 리스트를 반환받을 수 있습니다.

In [ ]:
# 텍스트를 문자열 리스트로 분할

# 아래에 코드를 작성하세요


## 4. CharacterTextSplitter와 비교

동일한 파라미터로 `CharacterTextSplitter`와 `RecursiveCharacterTextSplitter`의 결과를 비교합니다.

In [ ]:
# ============================================================
# TODO: CharacterTextSplitter와 RecursiveCharacterTextSplitter의 분할 결과를 비교해보세요
# 힌트: 동일한 파라미터로 두 Splitter를 생성하고 split_text()로 분할한 후 개수와 평균 크기를 비교
# 예상 결과: 두 방식의 청크 개수와 평균 크기가 출력됩니다
# ============================================================

from langchain_text_splitters import CharacterTextSplitter

# 1단계: CharacterTextSplitter로 분할

# 2단계: RecursiveCharacterTextSplitter로 분할


> **실무 팁**: 한글 문서에서는 `separators`에 `". "`, `"? "`, `"! "`를 추가하면 문장 경계를 더 잘 인식합니다. 영문 기준 `chunk_size=1000`이라면 한글은 `chunk_size=800`으로 줄이는 것이 좋습니다.

## 5. 사용자 정의 구분자

기본 구분자 외에 한글 문서에 적합한 구분자를 추가할 수 있습니다.

In [23]:
# ============================================================
# TODO: 한글 문장 부호를 포함한 사용자 정의 separators를 RecursiveCharacterTextSplitter에 추가해보세요
# 힌트: separators=["\n\n", "\n", ". ", "? ", "! ", " ", ""]
# 예상 결과: 기본 구분자와 청크 개수 비교 결과가 출력됩니다
# ============================================================

# 1단계: 한글 문서에 적합한 구분자 정의
# - 한글 문장 부호(". ", "? ", "! ")를 추가해서 문장 단위 분할 가능

# 2단계: 분할 실행

korean_text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=250,
  chunk_overlap=50,
  separators=[
    "\n\n",
    "\n",
    ".",
    "?",
    "!"
  ]
)

korean_chunks = korean_text_splitter.split_text(file)

korean_chunks[3]


'. 이는 핵심 인프라 공격을 유예하며 협상을 진행하겠다고 밝힌 시한을 애초 예고했던 6일에서 7일로 하루 연장하겠다는 뜻으로 해석된다.트럼프 대통령은 이날 월스트리트저널(WSJ) 인터뷰에서 협상 시한을 오는 7일 저녁으로 제시했다.트럼프 대통령은 인터뷰에서 "만약 그들이 화요일(7일) 저녁까지 아무런 조치를 취하지 않는다면, 발전소는 하나도 남지 않을 것이고 다리도 하나도 서 있지 않을 것"이라고 말했다.'

## 6. 청크 크기 실험

다양한 `chunk_size` 값을 테스트하여 적절한 크기를 찾아봅니다.

## 핵심 정리 및 마무리

### 문서 유형별 권장 chunk_size

| 문서 유형 | chunk_size | chunk_overlap |
|----------|:---------:|:-------------:|
| 짧은 Q&A, 뉴스 | 500 | 50 |
| 일반 문서, 블로그 | 1000 | 100 |
| 기술 문서, 논문 | 1500 | 150 |
| 책, 매뉴얼 | 2000 | 200 |

### 한글 문서 팁
> 한글은 영어보다 토큰 수가 많아요. 영문 기준 `chunk_size=1000`이라면 한글 문서는 `chunk_size=800` 정도로 조정하는 게 좋아요. 마침표나 물음표를 구분자에 추가하면 더 자연스럽게 분할돼요.

---

## 다음 예고

문자(character) 기반 분할에서 **토큰(token)** 기반 분할로 넘어가요.

- **03-TokenTextSplitter**: LLM 토큰 제한을 정확히 준수하는 분할 방식
- **04-SemanticChunker**: 임베딩 유사도로 의미 단위를 찾는 고급 분할